# Proposal A: reproduce the IDS baseline
**Abdulstar Ahmed Kareem · MyPhDThesis**

Use **Runtime → Change runtime type → CPU (no accelerator)**, connect, then **Runtime → Run all**.

This notebook loads the standard and adversarially trained RLN checkpoints from TabularBench, evaluates the native CTU-13 Neris test records and runs a small constrained-attack check. It creates a separate Python 3.8.20 environment; your Colab notebook keeps its current Python and packages.

**First milestone:** reproduce the two confusion matrices. These are recorded, aggregated network features. The notebook does not send attack traffic or replay packets. Checkpoint-compatible scaling uses all feature rows, so a later controlled training study needs train-only preprocessing and separate validation/test data.

Expected clean counts: **55,082 test records; 54,675 benign; 407 malicious.** Standard RLN detects 398 and raises 88 false alarms. Adversarially trained RLN detects 396 and raises 42 false alarms.


## 1. Open the project and choose a small first run
The defaults run clean evaluation plus an **eight-record smoke test**. Leave the full-budget option off for your first attempt. Each execution creates new run directories and preserves earlier results.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

RUN_SMOKE_TEST = True #@param {type:"boolean"}
SMOKE_RECORDS = 8 #@param {type:"integer"}
RUN_FULL_BUDGET = False #@param {type:"boolean"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
REPO_REF = "main"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis")).resolve()
HOST_PYTHON = sys.executable
HOST_VERSION = sys.version
if SMOKE_RECORDS < 1:
    raise ValueError("SMOKE_RECORDS must be at least 1.")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} exists but is not a git checkout. Choose a new workspace.")
# Reuse existing work without git reset, checkout or pull; record exactly what ran.
for required in ["scripts/bootstrap.py", "baseline/run_baseline.py"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the completed repository.")

def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("colab")
SESSION.mkdir(parents=True, exist_ok=False)
sha = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--verify", "HEAD"], text=True, capture_output=True)
project_commit = sha.stdout.strip() if sha.returncode == 0 else "uncommitted-workspace"
source_paths = [p for folder in ["baseline", "scripts"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/01_Baseline_Reproduction.ipynb"]
source_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths}
context = {"project_repository": REPO_URL, "project_commit": project_commit,
           "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
           "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
           "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "source_sha256": source_hashes,
           "run_smoke_test": RUN_SMOKE_TEST, "smoke_records": SMOKE_RECORDS,
           "run_full_budget": RUN_FULL_BUDGET}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
SNAPSHOT = SESSION / "source_snapshot"
for p in source_paths:
    target = SNAPSHOT / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    # Preserve a prior log if a cell is repeated.
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Fix the error, then rerun this cell.")

print("Notebook Python:", sys.version.split()[0])
print("Project commit:", project_commit)
print("Results folder:", SESSION)


## 2. Install the baseline environment
The first setup downloads dependencies and the pinned upstream source. Let this cell finish; it may take several minutes. CPU mode is sufficient. You do not need to restart or downgrade the notebook runtime.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
shutil.copy2(ROOT / ".cache/setup_report.json", SESSION / "setup_report.json")
print("Notebook interpreter unchanged. Baseline interpreter:", PYTHON)


## 3. Download and verify the dataset and two checkpoints
The dataset is about 348 MB. Existing verified downloads are reused within the current runtime. SHA-256 checks guard against the wrong files. No token is needed for these public assets.


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
shutil.copy2(ROOT / "assets/download_manifest.json", SESSION / "download_manifest.json")


## 4. Evaluate both models on the complete native test partition
This evaluates both standard and adversarially trained RLN. It does not train a new model. A full test partition is preferable to choosing an arbitrary 10,000 records.


In [ ]:
CLEAN_DIR = SESSION / unique_name("clean")
run_command([PYTHON, ROOT / "baseline/run_baseline.py", "--repo", ROOT / "vendor/tabularbench",
             "--assets", ROOT / "assets", "--out", CLEAN_DIR], "clean.log")
run_command([PYTHON, ROOT / "scripts/summarize_results.py", CLEAN_DIR], "clean_summary.log")


## 5. Inspect the scores and confusion matrices
Recall measures detected malicious records; false-positive rate measures benign records incorrectly flagged. Compare both. Accuracy alone is misleading because benign traffic dominates this test set.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

clean_report = json.loads((CLEAN_DIR / "results.json").read_text())
display(pd.read_csv(CLEAN_DIR / "summary.csv"))
expected_counts = {"default": (54587, 88, 9, 398), "madry": (54633, 42, 11, 396)}
for name, expected in expected_counts.items():
    actual = tuple(clean_report["models"][name]["clean"][k] for k in ["tn", "fp", "fn", "tp"])
    if actual != expected:
        raise RuntimeError(f"Baseline count mismatch for {name}: {actual}. Investigate before modifying the model.")
print("Both clean confusion matrices match the verified reference.")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ["default", "madry"]):
    m = clean_report["models"][name]["clean"]
    matrix = [[m["tn"], m["fp"]], [m["fn"], m["tp"]]]
    ax.imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{matrix[i][j]:,}", ha="center", va="center",
                    color="white" if matrix[i][j] > m["tn"] / 2 else "black")
    ax.set_xticks([0, 1], labels=["Benign", "Malicious"])
    ax.set_yticks([0, 1], labels=["Benign", "Malicious"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Standard RLN" if name == "default" else "Adversarially trained RLN")
fig.tight_layout()
fig.savefig(CLEAN_DIR / "confusion_matrices.png", dpi=180)
plt.show()


## 6. Run the constrained-attack smoke test
Both models are evaluated on the **same malicious records**, selected independently of model predictions. Default: 8 records, scaled L2 budget 0.5, 10 CAPGD steps, 2 MOEVA generations and 20 offspring.

This checks execution, not a published robust score. Zero evasions on this tiny test do not prove security. CAA may return unchanged originals when no accepted successful perturbation is found. Final candidate validity is not the validity rate of every internal proposal.


In [ ]:
if RUN_SMOKE_TEST:
    SMOKE_DIR = SESSION / unique_name("smoke")
    run_command([PYTHON, ROOT / "baseline/run_baseline.py", "--repo", ROOT / "vendor/tabularbench",
                 "--assets", ROOT / "assets", "--out", SMOKE_DIR, "--attack",
                 "--n-attack", str(SMOKE_RECORDS), "--steps", "10", "--generations", "2",
                 "--offspring", "20", "--eps", "0.5", "--seed", "0"], "smoke.log")
    run_command([PYTHON, ROOT / "scripts/summarize_results.py", SMOKE_DIR], "smoke_summary.log")
    display(pd.read_csv(SMOKE_DIR / "summary.csv"))
    for model in ["default", "madry"]:
        print(model, "candidate audit:")
        display(pd.read_csv(SMOKE_DIR / f"{model}_attack_audit.csv"))
else:
    print("Smoke test disabled.")


## 7. Optional expanded attack evaluation — disabled by default
Enable `RUN_FULL_BUDGET` in cell 1 only after the clean and smoke runs work. This attacks **all eligible malicious test records**, with 100 generations and 100 offspring. Runtime is not yet measured and may be substantially longer.

These main search-budget settings follow the current benchmark defaults, but the original sampling/repetition/aggregation protocol still needs reconciliation before claiming reproduction of the paper's constrained-attack score. Failed search does not certify robustness.


In [ ]:
if RUN_FULL_BUDGET:
    FULL_DIR = SESSION / unique_name("full_budget")
    run_command([PYTHON, ROOT / "baseline/run_baseline.py", "--repo", ROOT / "vendor/tabularbench",
                 "--assets", ROOT / "assets", "--out", FULL_DIR, "--attack", "--n-attack", "0",
                 "--steps", "10", "--generations", "100", "--offspring", "100",
                 "--eps", "0.5", "--seed", "0"], "full_budget.log")
    run_command([PYTHON, ROOT / "scripts/summarize_results.py", FULL_DIR], "full_summary.log")
    display(pd.read_csv(FULL_DIR / "summary.csv"))
else:
    print("Full-budget experiment skipped. The default run ends with the smoke test.")


## 8. Save your results before leaving Colab
The ZIP contains all runs in this session, CSV predictions, JSON settings, logs, plots, audited attack arrays and the source snapshot. It excludes downloaded datasets, checkpoints and Python packages.

`SAVE_TO_DRIVE=True` copies the ZIP to **My Drive/MyPhDThesis/results** after the normal Google authorization prompt. `DOWNLOAD_RESULTS=True` also starts a browser download. If downloads are blocked, use Colab's Files panel and download the ZIP from `MyPhDThesis/exports`.

Colab's runtime storage is temporary. Saving this notebook to GitHub does not save the experiment outputs.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## What comes next in the research?
1. Complete and audit the stronger baseline attack evaluation.
2. Establish train-only preprocessing, separate tuning/calibration data, and a sealed test partition for new training experiments.
3. Define one supported additional attacker capability, including dependencies and measurable costs; inspect whether it changes defense rankings.
4. Only then implement and compare the proposed capability-aware training rule, with held-out capability tests.

See [NEXT_EXPERIMENTS.md](https://github.com/Abdulstar/MyPhDThesis/blob/main/docs/NEXT_EXPERIMENTS.md). A new training rule and validated findings are the intended research outcomes.

Sources: [TabularBench](https://arxiv.org/abs/2408.07579), [CAA](https://arxiv.org/abs/2406.00775), [upstream repository](https://github.com/serval-uni-lu/tabularbench), [Colab FAQ](https://research.google.com/colaboratory/faq.html).
